# 1. Imports and Environment Setup

In [1]:
import os
import re
import json
import time
import textwrap
from pathlib import Path

import requests
import pandas as pd
from dotenv import load_dotenv
from huggingface_hub import InferenceClient # Hugging Face Inference API client
from openai import OpenAI # OpenAI API client

# Load environment variables from a local .env file if present
load_dotenv()

# Project paths
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

# Create common folders if they do not already exist
DATA_DIR.mkdir(exist_ok=True)
OUTPUTS_DIR.mkdir(exist_ok=True)

# Environment variables
HF_TOKEN = os.getenv("HF_TOKEN", "")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "")
LLAMA_MODEL = os.getenv("LLAMA_MODEL", "")


# Basic environment checks
print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)
print("Outputs directory:", OUTPUTS_DIR)


if HF_TOKEN:
    print("HF_TOKEN loaded.")
else:
    print("HF_TOKEN not found in environment.")

if OPENAI_API_KEY:
    print("OPENAI_API_KEY loaded.")
else:
    print("OPENAI_API_KEY not found in environment.")

if OPENAI_MODEL:
    print("OPENAI_MODEL:", OPENAI_MODEL)
else:    print("OPENAI_MODEL not set.")

if LLAMA_MODEL:
    print("LLAMA_MODEL:", LLAMA_MODEL)
else:
    print("LLAMA_MODEL not set.")

Project root: c:\Users\Yuna\PolishCV
Data directory: c:\Users\Yuna\PolishCV\data
Outputs directory: c:\Users\Yuna\PolishCV\outputs
HF_TOKEN loaded.
OPENAI_API_KEY loaded.
OPENAI_MODEL: gpt-5.4
LLAMA_MODEL: meta-llama/Llama-3.2-1B-Instruct


# 2.  AI Configuration

In [2]:
PROVIDER = "huggingface"   # current plan: use Hugging Face for both models

MODEL_OPTIONS = {
    "openai": OPENAI_MODEL if OPENAI_MODEL else "gpt-5.4",
    "llama": LLAMA_MODEL if LLAMA_MODEL else "meta-llama/Llama-3.2-3B-Instruct"
}

# Generation settings
GENERATION_CONFIG = {
    "max_new_tokens": 100,
    "temperature": 0.3,
    "top_p": 0.9,
    "do_sample": True,
    "return_full_text": False
}


# Input limits
MIN_RESUME_CHARS = 100
MIN_JOB_DESCRIPTION_CHARS = 100
MAX_INPUT_CHARS = 12000


# File paths
TEST_CASES_PATH = DATA_DIR / "test_cases.json"
VERIFICATION_CASES_PATH = DATA_DIR / "verification_cases.json"
SAMPLE_INPUTS_PATH = DATA_DIR / "sample_inputs.json"
EVAL_RESULTS_PATH = OUTPUTS_DIR / "eval_results.csv"
SAMPLE_OUTPUTS_PATH = OUTPUTS_DIR / "sample_model_outputs.json"


# Evaluation settings
SUPPORTED_TASKS = [
    "resume_feedback",
    "resume_tailor"
]

DEFAULT_TASK = "resume_feedback"

RUBRIC_DIMENSIONS = [
    "ats_alignment",
    "factual_faithfulness",
    "relevance_to_job_description",
    "clarity_professionalism",
    "usefulness_of_feedback"
]


# Safety / trustworthiness settings
ANTI_HALLUCINATION_RULES = {
    "no_new_jobs": True,
    "no_new_dates": True,
    "no_new_metrics": True,
    "no_new_certifications": True,
    "preserve_user_facts": True
}

print("Provider:", PROVIDER)
print("Available models:", MODEL_OPTIONS)
print("Supported tasks:", SUPPORTED_TASKS)
print("Evaluation results path:", EVAL_RESULTS_PATH)

Provider: huggingface
Available models: {'openai': 'gpt-5.4', 'llama': 'meta-llama/Llama-3.2-1B-Instruct'}
Supported tasks: ['resume_feedback', 'resume_tailor']
Evaluation results path: c:\Users\Yuna\PolishCV\outputs\eval_results.csv


# 3. Utility Functions

In [3]:
#Utility helper functions for the PolishCV project

from typing import Any, Dict, List, Optional


def clean_text(text: str) -> str:
    """
    Clean and normalize input text.

    What it should do:
    - handle None or empty input safely
    - strip leading/trailing whitespace
    - normalize repeated spaces
    - normalize repeated blank lines
    """
    pass


def validate_inputs(resume_text: str, job_description: str) -> Dict[str, Any]:
    """
    Validate resume and job description inputs.

    Returns a dictionary such as:
    {
        "is_valid": True/False,
        "errors": [ ... ]
    }

    Checks to include:
    - resume is not empty
    - job description is not empty
    - both meet minimum character count
    - both stay under max input limit
    """
    pass


def truncate_text(text: str, max_chars: int = MAX_INPUT_CHARS) -> str:
    """
    Truncate text if it exceeds the allowed maximum length.
    """
    pass


def extract_keywords_from_jd(job_description: str, top_n: int = 20) -> List[str]:
    """
    Extract simple keywords from a job description.

    This can be a lightweight keyword extractor for:
    - skills
    - tools
    - technologies
    - repeated important terms

    Keep this simple for now.
    """
    pass


def safe_json_loads(text: str) -> Optional[Dict[str, Any]]:
    """
    Safely parse JSON returned by a model.

    Returns:
    - parsed dictionary if successful
    - None if parsing fails
    """
    pass


def parse_model_output(raw_output: str) -> Dict[str, Any]:
    """
    Convert raw model output into a structured dictionary.

    Expected target structure could look like:
    {
        "summary": "",
        "rewritten_experience": [],
        "missing_keywords": [],
        "feedback": [],
        "gap_suggestions": [],
        "risk_flags": []
    }

    For now, this is a fallback parser skeleton.
    """
    pass


def format_bullet_list(items: List[str]) -> str:
    """
    Format a list of strings as bullet points for display.
    """
    pass


def compute_latency(start_time: float, end_time: float) -> float:
    """
    Compute elapsed time in seconds for a model request.
    """
    pass


def load_json_file(file_path: Path) -> Any:
    """
    Load JSON data from a file path.
    """
    pass


def save_json_file(data: Any, file_path: Path) -> None:
    """
    Save JSON data to a file path.
    """
    pass


def append_results_to_csv(row: Dict[str, Any], file_path: Path) -> None:
    """
    Append one evaluation result row to a CSV file.

    Useful for logging:
    - case_id
    - model_name
    - task
    - before_score
    - after_score
    - score_delta
    - latency
    - notes
    """
    pass

#  4. System Prompts

In [4]:
# Purpose: Prompt templates for the PolishCV project

def build_system_instruction() -> str:
    """
    Shared system-style instruction used across prompt types.
    """
    return """
You are an experienced technical recruiter and resume reviewer focused on entry-level software engineering roles.

Your job is to improve resumes in a helpful, professional, and trustworthy way.

Important rules:
1. Do not invent facts.
2. Do not add new companies, job titles, dates, certifications, degrees, or metrics unless they are explicitly provided by the user.
3. Do not exaggerate the candidate's experience.
4. Tailor suggestions to the provided job description only.
5. Prefer clear, concise, ATS-friendly language.
6. Explain your suggestions in a way that a human user can review and verify.
7. If important information is missing, say so instead of making assumptions.
""".strip()


def build_output_format_instruction() -> str:
    """
    Instruction telling the model to return structured JSON.
    """
    return """
Return your answer as valid JSON with the following keys:
{
  "summary": "short overview of your assessment",
  "rewritten_experience": ["list of improved resume bullet points or revised lines"],
  "missing_keywords": ["list of important keywords or skills missing from the resume"],
  "feedback": ["list of concrete suggestions for improvement"],
  "gap_suggestions": ["list of suggestions for addressing weak areas or experience gaps"],
  "risk_flags": ["list of possible trustworthiness or accuracy concerns the user should review"]
}

Rules for formatting:
- Return JSON only.
- Do not include markdown.
- Do not include code fences.
- If a field has no content, return an empty list or empty string.
""".strip()


def build_resume_feedback_prompt(resume_text: str, job_description: str) -> str:
    """
    Build a prompt for reviewing the user's current resume
    against a target job description.
    """
    system_instruction = build_system_instruction()
    output_instruction = build_output_format_instruction()

    return f"""
{system_instruction}

Task:
Review the user's resume for an entry-level software engineering job.
Compare it against the target job description and provide ATS-friendly feedback.

Resume:
{resume_text}

Target Job Description:
{job_description}

What to do:
- identify strengths
- identify missing or weak keywords and skills
- point out unclear, weak, or overly generic wording
- suggest improvements that better align the resume with the job description
- suggest ways to strengthen weak areas without inventing experience
- flag anything that might be inaccurate, misleading, or too vague

{output_instruction}
""".strip()


def build_resume_tailor_prompt(resume_text: str, job_description: str) -> str:
    """
    Build a prompt for tailoring the resume to a job description.
    """
    system_instruction = build_system_instruction()
    output_instruction = build_output_format_instruction()

    return f"""
{system_instruction}

Task:
Tailor the user's resume for the target entry-level software engineering role.

Resume:
{resume_text}

Target Job Description:
{job_description}

What to do:
- rewrite parts of the resume to better align with the job description
- improve clarity, relevance, and ATS-friendly wording
- preserve the user's original facts
- do not invent experience or metrics
- highlight important missing keywords
- provide feedback explaining the most important changes

{output_instruction}
""".strip()


def build_gap_suggestions_prompt(resume_text: str, job_description: str) -> str:
    """
    Optional lightweight prompt for suggestions on how a user
    could strengthen weak areas without fabricating experience.
    """
    system_instruction = build_system_instruction()
    output_instruction = build_output_format_instruction()

    return f"""
{system_instruction}

Task:
Suggest realistic ways the user could strengthen weak areas in the resume
for the target entry-level software engineering role.

Resume:
{resume_text}

Target Job Description:
{job_description}

What to do:
- identify possible experience or skill gaps
- suggest realistic next steps such as projects, internships, volunteer work,
  open-source work, research, coursework, or certifications
- do not claim the user already has these experiences
- do not rewrite the entire resume
- keep suggestions practical and appropriate for a student or recent graduate

{output_instruction}
""".strip()


def build_prompt(task_name: str, resume_text: str, job_description: str) -> str:
    """
    Dispatch function for selecting the correct prompt template.
    """
    if task_name == "resume_feedback":
        return build_resume_feedback_prompt(resume_text, job_description)
    elif task_name == "resume_tailor":
        return build_resume_tailor_prompt(resume_text, job_description)
    elif task_name == "gap_suggestions":
        return build_gap_suggestions_prompt(resume_text, job_description)
    else:
        raise ValueError(f"Unsupported task_name: {task_name}")

# 5. Model Inference Functions

In [5]:
# Model inference functions for the PolishCV project

from typing import Tuple

def get_hf_client() -> InferenceClient:
    """
    Create a Hugging Face inference client using the token from .env
    """
    if not HF_TOKEN:
        raise ValueError("HF_TOKEN is missing. Please add it to your .env file.")
    return InferenceClient(api_key=HF_TOKEN)



def call_hf_model(model_id: str, prompt: str) -> Dict[str, Any]:
    """
    Call a Hugging Face model using InferenceClient and return a structured response.
    """
    client = get_hf_client()
    start_time = time.time()

    try:
        completion = client.chat.completions.create(
            model=model_id,
            messages=[
                {"role": "user", "content": prompt}
            ],
            max_tokens=GENERATION_CONFIG.get("max_new_tokens", 500),
            temperature=GENERATION_CONFIG.get("temperature", 0.3),
            top_p=GENERATION_CONFIG.get("top_p", 0.9),
        )
        end_time = time.time()

        generated_text = ""
        if completion and getattr(completion, "choices", None):
            generated_text = completion.choices[0].message.content or ""

        if not generated_text.strip():
            return {
                "success": False,
                "model_id": model_id,
                "raw_text": "",
                "latency_seconds": compute_latency(start_time, end_time),
                "error": "Model response was empty."
            }

        return {
            "success": True,
            "model_id": model_id,
            "raw_text": generated_text.strip(),
            "latency_seconds": compute_latency(start_time, end_time),
            "error": None
        }

    except Exception as exc:
        end_time = time.time()
        return {
            "success": False,
            "model_id": model_id,
            "raw_text": "",
            "latency_seconds": compute_latency(start_time, end_time),
            "error": str(exc)
        }

def get_openai_client() -> OpenAI:
    """
    Create an OpenAI client using the API key from .env
    """
    if not OPENAI_API_KEY:
        raise ValueError("OPENAI_API_KEY is missing. Please add it to your .env file.")
    return OpenAI(api_key=OPENAI_API_KEY)


def call_openai_model(model_id: str, prompt: str) -> Dict[str, Any]:
    """
    Call an OpenAI model and return a structured response.
    """
    client = get_openai_client()
    start_time = time.time()

    try:
        completion = client.chat.completions.create(
            model=model_id,
            messages=[
                {"role": "user", "content": prompt}
            ],
            max_completion_tokens=GENERATION_CONFIG.get("max_new_tokens", 500),
            temperature=GENERATION_CONFIG.get("temperature", 0.3),
            top_p=GENERATION_CONFIG.get("top_p", 0.9),
        )
        end_time = time.time()

        generated_text = ""
        if completion and getattr(completion, "choices", None):
            generated_text = completion.choices[0].message.content or ""

        if not generated_text.strip():
            return {
                "success": False,
                "model_id": model_id,
                "raw_text": "",
                "latency_seconds": compute_latency(start_time, end_time),
                "error": "Model response was empty."
            }

        return {
            "success": True,
            "model_id": model_id,
            "raw_text": generated_text.strip(),
            "latency_seconds": compute_latency(start_time, end_time),
            "error": None
        }

    except Exception as exc:
        end_time = time.time()
        return {
            "success": False,
            "model_id": model_id,
            "raw_text": "",
            "latency_seconds": compute_latency(start_time, end_time),
            "error": str(exc)
        }


def run_openai(prompt: str) -> Dict[str, Any]:
    model_id = MODEL_OPTIONS["openai"]
    return call_openai_model(model_id=model_id, prompt=prompt)

def run_llama(prompt: str) -> Dict[str, Any]:
    """
    Run the configured Llama model.
    """
    model_id = MODEL_OPTIONS["llama"]
    return call_hf_model(model_id=model_id, prompt=prompt)


def run_selected_model(model_name: str, prompt: str) -> Dict[str, Any]:
    """
    Dispatch inference based on the selected model name.
    """
    model_name = model_name.lower().strip()

    if model_name == "openai":
        return run_openai(prompt)
    elif model_name == "llama":
        return run_llama(prompt)
    else:
        raise ValueError(f"Unsupported model_name: {model_name}")


def generate_for_task(model_name: str, task_name: str, resume_text: str, job_description: str) -> Dict[str, Any]:
    """
    End-to-end helper function:
    - builds the task prompt
    - calls the selected model
    - returns the raw model response
    """
    prompt = build_prompt(
        task_name=task_name,
        resume_text=resume_text,
        job_description=job_description
    )

    result = run_selected_model(model_name=model_name, prompt=prompt)
    result["task_name"] = task_name
    result["prompt"] = prompt
    return result

# 6. ATS Alignment Scoring

In [6]:
# Purpose: ATS-alignment scoring functions for the PolishCV project


STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for", "from", "in",
    "is", "it", "of", "on", "or", "that", "the", "to", "with", "will",
    "this", "your", "you", "our", "their", "they", "we", "was", "were",
    "has", "have", "had", "but", "not", "if", "into", "than", "then",
    "so", "such", "using", "use", "used", "about", "can", "should"
}

COMMON_TECH_KEYWORDS = {
    "python", "java", "javascript", "typescript", "c++", "c", "sql", "html",
    "css", "react", "node", "node.js", "flask", "django", "git", "github",
    "aws", "docker", "kubernetes", "linux", "api", "rest", "tensorflow",
    "pytorch", "pandas", "numpy", "machine learning", "data structures",
    "algorithms", "oop", "debugging", "testing", "streamlit"
}

ACTION_VERBS = {
    "built", "developed", "designed", "implemented", "created", "optimized",
    "improved", "deployed", "tested", "analyzed", "collaborated", "led",
    "automated", "engineered", "debugged", "refactored", "integrated"
}

COMMON_RESUME_SECTIONS = {
    "education", "experience", "projects", "skills", "summary"
}


def normalize_text(text: str) -> str:
    """
    Lowercase and lightly normalize text for matching.
    """
    if not text:
        return ""
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


def tokenize_text(text: str) -> List[str]:
    """
    Tokenize text into simple lowercase words.
    """
    text = normalize_text(text)
    tokens = re.findall(r"[a-zA-Z0-9\+\.\#\-]+", text)
    return tokens


def get_keyword_set(text: str) -> set:
    """
    Convert text into a basic keyword set after stopword filtering.
    """
    tokens = tokenize_text(text)
    keywords = {
        token for token in tokens
        if token not in STOPWORDS and len(token) > 1
    }
    return keywords


def extract_important_jd_keywords(job_description: str, top_n: int = 20) -> List[str]:
    """
    Extract a simple list of important keywords from the job description.

    Strategy:
    - tokenize
    - remove stopwords
    - count frequency
    - prioritize tech-related keywords when present
    """
    tokens = tokenize_text(job_description)
    filtered = [t for t in tokens if t not in STOPWORDS and len(t) > 1]

    counts: Dict[str, int] = {}
    for token in filtered:
        counts[token] = counts.get(token, 0) + 1

    sorted_tokens = sorted(
        counts.items(),
        key=lambda item: (
            item[0] not in COMMON_TECH_KEYWORDS,   # prefer known tech terms first
            -item[1],                              # then by frequency
            item[0]
        )
    )

    return [token for token, _ in sorted_tokens[:top_n]]


def keyword_overlap_score(resume_text: str, job_description: str, top_n: int = 20) -> float:
    """
    Score overlap between important JD keywords and resume keywords.
    Returns a value from 0 to 100.
    """
    jd_keywords = set(extract_important_jd_keywords(job_description, top_n=top_n))
    resume_keywords = get_keyword_set(resume_text)

    if not jd_keywords:
        return 0.0

    overlap = jd_keywords.intersection(resume_keywords)
    score = (len(overlap) / len(jd_keywords)) * 100
    return round(score, 2)


def tech_keyword_score(resume_text: str, job_description: str) -> float:
    """
    Measure coverage of common technical keywords that appear in the JD.
    Returns a value from 0 to 100.
    """
    jd_text = normalize_text(job_description)
    resume_text_norm = normalize_text(resume_text)

    jd_tech_terms = [kw for kw in COMMON_TECH_KEYWORDS if kw in jd_text]

    if not jd_tech_terms:
        return 100.0

    matched = [kw for kw in jd_tech_terms if kw in resume_text_norm]
    score = (len(matched) / len(jd_tech_terms)) * 100
    return round(score, 2)


def action_verb_score(resume_text: str) -> float:
    """
    Measure how much the resume uses action-oriented language.
    Returns a value from 0 to 100.
    """
    resume_tokens = set(tokenize_text(resume_text))
    matched = ACTION_VERBS.intersection(resume_tokens)

    if not ACTION_VERBS:
        return 0.0

    score = min((len(matched) / 8) * 100, 100)  # cap after a reasonable number
    return round(score, 2)


def section_coverage_score(resume_text: str) -> float:
    """
    Check whether common resume sections are present.
    Returns a value from 0 to 100.
    """
    resume_text_norm = normalize_text(resume_text)
    present_sections = [section for section in COMMON_RESUME_SECTIONS if section in resume_text_norm]

    score = (len(present_sections) / len(COMMON_RESUME_SECTIONS)) * 100
    return round(score, 2)


def compute_ats_score(resume_text: str, job_description: str) -> Dict[str, Any]:
    """
    Compute a simple ATS-alignment score with component breakdown.

    Weighted components:
    - keyword overlap: 45%
    - technical keyword coverage: 30%
    - action verbs: 15%
    - section coverage: 10%
    """
    overlap = keyword_overlap_score(resume_text, job_description)
    tech = tech_keyword_score(resume_text, job_description)
    action = action_verb_score(resume_text)
    sections = section_coverage_score(resume_text)

    overall = (
        0.45 * overlap +
        0.30 * tech +
        0.15 * action +
        0.10 * sections
    )

    return {
        "overall_score": round(overall, 2),
        "keyword_overlap_score": overlap,
        "tech_keyword_score": tech,
        "action_verb_score": action,
        "section_coverage_score": sections,
        "important_jd_keywords": extract_important_jd_keywords(job_description, top_n=20)
    }


def compare_scores(original_resume: str, revised_resume: str, job_description: str) -> Dict[str, Any]:
    """
    Compare ATS-alignment scores before and after revision.
    """
    before = compute_ats_score(original_resume, job_description)
    after = compute_ats_score(revised_resume, job_description)

    delta = round(after["overall_score"] - before["overall_score"], 2)

    return {
        "before": before,
        "after": after,
        "score_delta": delta
    }


def score_band(score: float) -> str:
    """
    Map a numeric score to the project's interpretation bands.
    """
    if score >= 80:
        return "Excellent / Strong match"
    elif score >= 60:
        return "Good / Moderate match"
    return "Weak match"

In [7]:
# Simple scoring smoke test
sample_resume = """
Education
B.S. in Computer Science

Projects
Built a Streamlit app using Python and pandas.
Developed a REST API for a student project.

Skills
Python, Streamlit, Git, SQL
"""

sample_jd = """
We are hiring an entry-level software engineer with experience in Python,
REST APIs, Git, SQL, debugging, and teamwork. Experience with Streamlit is a plus.
"""

sample_score = compute_ats_score(sample_resume, sample_jd)
sample_score

{'overall_score': 52.07,
 'keyword_overlap_score': 35.71,
 'tech_keyword_score': 87.5,
 'action_verb_score': 25.0,
 'section_coverage_score': 60.0,
 'important_jd_keywords': ['debugging',
  'git',
  'python',
  'rest',
  'sql',
  'streamlit',
  'experience',
  'apis',
  'engineer',
  'entry-level',
  'hiring',
  'plus.',
  'software',
  'teamwork.']}

# 7. Evaluation Rubric

## Evaluation Rubric

This section defines the human evaluation rubric used to assess the quality and trustworthiness of model outputs.

Because resume improvement is not purely objective, ATS-alignment score alone is not enough. In addition to the automated score, the project uses human evaluation to judge whether the generated feedback and revisions are actually helpful, accurate, and appropriate.

Each model output will be rated on the following dimensions:

1. **ATS Alignment**  
   How well the revised resume appears to match the target job description in terms of relevant keywords, skills, and role-specific language.

2. **Factual Faithfulness**  
   Whether the model preserves the user’s original facts and avoids inventing experience, metrics, certifications, job titles, dates, or other unsupported claims.

3. **Relevance to Job Description**  
   Whether the feedback or rewrite is clearly aligned with the target software engineering role and focuses on the most important requirements in the job description.

4. **Clarity and Professionalism**  
   Whether the output is clear, well-written, professional, and appropriate for a resume or resume feedback context.

5. **Usefulness of Feedback**  
   Whether the feedback is concrete, actionable, and likely to help the user improve their resume.

### Rating Scale

Each dimension will be rated on a **1 to 5 scale**:

- **1 = Very poor**
- **2 = Poor**
- **3 = Acceptable**
- **4 = Good**
- **5 = Excellent**

### Suggested Interpretation

- A strong output should score well on both automated ATS alignment and human evaluation.
- A response that improves ATS score but introduces false or exaggerated claims should be rated poorly on factual faithfulness.
- Human evaluation is especially important for identifying hallucinations, vague advice, misleading wording, and low-value suggestions.

### Evaluation Procedure

For each test or verification case:

- run the same task on both Qwen and Llama
- review the structured output
- compute the ATS-alignment score before and after revision
- assign human ratings using the rubric above
- record notes about strengths, weaknesses, and possible trustworthiness concerns

This rubric supports the project’s goal of evaluating not only whether the model improves resume-job matching, but also whether the output remains accurate, trustworthy, and useful for human users.

In [8]:
# Optional rubric template for manual evaluation

RUBRIC_TEMPLATE = {
    "ats_alignment": None,
    "factual_faithfulness": None,
    "relevance_to_job_description": None,
    "clarity_professionalism": None,
    "usefulness_of_feedback": None,
    "notes": ""
}

RUBRIC_TEMPLATE

{'ats_alignment': None,
 'factual_faithfulness': None,
 'relevance_to_job_description': None,
 'clarity_professionalism': None,
 'usefulness_of_feedback': None,
 'notes': ''}

# 8. Load Test and Verification Data

## Load Test and Verification Data

This section loads the test and verification datasets used in the project.

For this project, the data is not a traditional training dataset. Instead, it is a set of evaluation cases designed to test how well the GenAI models perform on resume-related tasks.

Each evaluation case should include:

- a unique case ID
- resume text
- target job description
- task type
- optional notes about expected weaknesses or missing keywords

The project uses **two separate datasets**:

1. **Test data**  
   Used for early experiments, prompt refinement, debugging, and initial comparisons.

2. **Verification data**  
   Used only for the final evaluation of model performance.

Keeping these two datasets separate is important because the course instructions explicitly say to use test data for tuning and initial testing, and verification data for final accuracy estimates, without mixing them. :contentReference[oaicite:0]{index=0}

This helps make the evaluation more trustworthy and better aligned with the final report requirements.

In [9]:
# Load test and verification datasets for the PolishCV project

def load_evaluation_cases(file_path: Path) -> List[Dict[str, Any]]:
    """
    Load evaluation cases from a JSON file.

    Expected JSON format:
    [
      {
        "case_id": "test_001",
        "task_name": "resume_feedback",
        "resume_text": "...",
        "job_description": "...",
        "notes": "Optional notes"
      }
    ]
    """
    if not file_path.exists():
        print(f"Warning: File not found -> {file_path}")
        return []

    data = load_json_file(file_path)

    if not isinstance(data, list):
        raise ValueError(f"Expected a list of cases in {file_path}")

    return data


def validate_case_schema(case: Dict[str, Any]) -> Dict[str, Any]:
    """
    Validate the minimum schema for one evaluation case.
    """
    required_fields = ["case_id", "task_name", "resume_text", "job_description"]
    errors = []

    for field in required_fields:
        if field not in case or not str(case[field]).strip():
            errors.append(f"Missing or empty field: {field}")

    if case.get("task_name") not in SUPPORTED_TASKS:
        errors.append(f"Unsupported task_name: {case.get('task_name')}")

    return {
        "is_valid": len(errors) == 0,
        "errors": errors
    }


def summarize_case_set(cases: List[Dict[str, Any]], label: str = "dataset") -> pd.DataFrame:
    """
    Build a small summary table for a set of evaluation cases.
    """
    rows = []

    for case in cases:
        rows.append({
            "dataset": label,
            "case_id": case.get("case_id", ""),
            "task_name": case.get("task_name", ""),
            "resume_chars": len(case.get("resume_text", "")),
            "job_description_chars": len(case.get("job_description", "")),
            "has_notes": bool(case.get("notes", "").strip()) if isinstance(case.get("notes", ""), str) else False
        })

    return pd.DataFrame(rows)


# Load both datasets
test_cases = load_evaluation_cases(TEST_CASES_PATH)
verification_cases = load_evaluation_cases(VERIFICATION_CASES_PATH)

print(f"Loaded {len(test_cases)} test cases.")
print(f"Loaded {len(verification_cases)} verification cases.")

# Validate schemas
test_validation_results = [validate_case_schema(case) for case in test_cases]
verification_validation_results = [validate_case_schema(case) for case in verification_cases]

invalid_test_cases = [i for i, result in enumerate(test_validation_results) if not result["is_valid"]]
invalid_verification_cases = [i for i, result in enumerate(verification_validation_results) if not result["is_valid"]]

print("Invalid test case indexes:", invalid_test_cases)
print("Invalid verification case indexes:", invalid_verification_cases)

# Show summaries
test_summary_df = summarize_case_set(test_cases, label="test")
verification_summary_df = summarize_case_set(verification_cases, label="verification")

display(test_summary_df)
display(verification_summary_df)

Loaded 0 test cases.
Loaded 0 verification cases.
Invalid test case indexes: []
Invalid verification case indexes: []


""


""


# 9. Single Example Run

### run one case through one or both models and inspect the output.

In [10]:
# Purpose: Run one sample case through the full PolishCV pipeline
def get_revised_resume_text(parsed_output: Dict[str, Any], original_resume_text: str) -> str:
    """
    Build a revised resume text string from parsed model output.

    For now, this uses the rewritten_experience field if available.
    If no rewritten content is present, it falls back to the original resume.
    """
    rewritten_experience = parsed_output.get("rewritten_experience", [])

    if isinstance(rewritten_experience, list) and rewritten_experience:
        return "\n".join(str(item).strip() for item in rewritten_experience if str(item).strip())

    return original_resume_text


def run_single_case(case: Dict[str, Any], model_name: str) -> Dict[str, Any]:
    """
    Run one evaluation case through:
    - input validation
    - prompt generation
    - model inference
    - output parsing
    - ATS score comparison
    """
    case_id = case.get("case_id", "unknown_case")
    task_name = case.get("task_name", DEFAULT_TASK)
    resume_text = clean_text(case.get("resume_text", ""))
    job_description = clean_text(case.get("job_description", ""))

    validation = validate_inputs(resume_text, job_description)
    if not validation["is_valid"]:
        return {
            "success": False,
            "case_id": case_id,
            "model_name": model_name,
            "task_name": task_name,
            "error": f"Input validation failed: {validation['errors']}"
        }

    inference_result = generate_for_task(
        model_name=model_name,
        task_name=task_name,
        resume_text=resume_text,
        job_description=job_description
    )

    if not inference_result["success"]:
        return {
            "success": False,
            "case_id": case_id,
            "model_name": model_name,
            "task_name": task_name,
            "error": inference_result["error"],
            "latency_seconds": inference_result.get("latency_seconds")
        }

    raw_output = inference_result["raw_text"]
    parsed_output = parse_model_output(raw_output)

    revised_resume_text = get_revised_resume_text(parsed_output, resume_text)
    score_comparison = compare_scores(resume_text, revised_resume_text, job_description)

    return {
        "success": True,
        "case_id": case_id,
        "model_name": model_name,
        "task_name": task_name,
        "resume_text": resume_text,
        "job_description": job_description,
        "raw_output": raw_output,
        "parsed_output": parsed_output,
        "revised_resume_text": revised_resume_text,
        "score_comparison": score_comparison,
        "latency_seconds": inference_result.get("latency_seconds"),
        "notes": case.get("notes", "")
    }


def display_single_case_result(result: Dict[str, Any]) -> None:
    """
    Display a single-case result in a readable way.
    """
    if not result["success"]:
        print("Run failed.")
        print("Case ID:", result.get("case_id"))
        print("Model:", result.get("model_name"))
        print("Task:", result.get("task_name"))
        print("Error:", result.get("error"))
        return

    print("=" * 80)
    print("Case ID:", result["case_id"])
    print("Model:", result["model_name"])
    print("Task:", result["task_name"])
    print("Latency (seconds):", result["latency_seconds"])
    print("=" * 80)

    print("\nOriginal Resume:\n")
    print(result["resume_text"])

    print("\nJob Description:\n")
    print(result["job_description"])

    print("\nParsed Model Output:\n")
    print(json.dumps(result["parsed_output"], indent=2))

    print("\nRevised Resume Text:\n")
    print(result["revised_resume_text"])

    print("\nATS Score Comparison:\n")
    print(json.dumps(result["score_comparison"], indent=2))

    if result.get("notes"):
        print("\nCase Notes:\n")
        print(result["notes"])

def parse_model_output(raw_output: str) -> Dict[str, Any]:
    return {
        "summary": "",
        "rewritten_experience": [],
        "missing_keywords": [],
        "feedback": [raw_output],
        "gap_suggestions": [],
        "risk_flags": []
    }

In [11]:
# # llama test with sample inputs

# sample_resume = """
# Computer Science student with experience building Python projects, using Git for version control,
# and working with SQL in coursework. Built a small web app for a class project and collaborated
# with teammates on debugging and testing.
# """

# sample_jd = """
# We are hiring an entry-level software engineer with experience in Python, SQL, Git,
# debugging, APIs, and teamwork. Candidates should be able to build and improve software systems.
# """

# # Build one prompt
# prompt = build_resume_feedback_prompt(sample_resume, sample_jd)

# # Choose which model to test
# llama_result = run_llama(prompt)   # or run_llama(prompt)

# print("Success:", llama_result["success"])
# print("Model ID:", llama_result["model_id"])
# print("Latency:", llama_result["latency_seconds"])

# if llama_result["success"]:
#     print("\nRAW MODEL OUTPUT:\n")
#     print(llama_result["raw_text"])

#     parsed = parse_model_output(llama_result["raw_text"])
#     print("\nPARSED OUTPUT:\n")
#     print(json.dumps(parsed, indent=2))
# else:
#     print("\nERROR:\n")
#     print(llama_result["error"])

In [12]:
# # openai test with sample inputs

# # Build one prompt
# prompt = build_resume_feedback_prompt(sample_resume, sample_jd)

# # Choose which model to test
# openai_result = run_openai(prompt)   # or run_openai(prompt)

# print("Success:", openai_result["success"])
# print("Model ID:", openai_result["model_id"])
# print("Latency:", openai_result["latency_seconds"])

# if openai_result["success"]:
#     print("\nRAW MODEL OUTPUT:\n")
#     print(openai_result["raw_text"])

#     parsed = parse_model_output(openai_result["raw_text"])
#     print("\nPARSED OUTPUT:\n")
#     print(json.dumps(parsed, indent=2))
# else:
#     print("\nERROR:\n")
#     print(openai_result["error"])

# 10. Batch Evaluation

## Batch Evaluation

This section runs multiple evaluation cases through the project pipeline and stores the results for later analysis.

The goal is to evaluate OpenAi and Llama on the same tasks in a consistent way. For each case, the notebook will:

- load the resume and job description
- run the selected model on the task
- parse the model output
- compute the ATS-alignment score before and after revision
- record metadata such as model name, task, latency, and errors
- save results for later comparison and reporting

This section is important because it produces the main experimental evidence used in the final report.

In [13]:
# Batch evaluation functions for the PolishCV project

def safe_get_score(score_comparison: Dict[str, Any], stage: str, key: str, default: float = 0.0) -> float:
    """
    Safely extract a score value from the score comparison dictionary.
    """
    try:
        return float(score_comparison.get(stage, {}).get(key, default))
    except (TypeError, ValueError):
        return default


def build_result_row(result: Dict[str, Any]) -> Dict[str, Any]:
    """
    Convert a single-case run result into a flat row for a DataFrame or CSV.
    """
    if not result["success"]:
        return {
            "case_id": result.get("case_id", ""),
            "model_name": result.get("model_name", ""),
            "task_name": result.get("task_name", ""),
            "success": False,
            "before_score": None,
            "after_score": None,
            "score_delta": None,
            "keyword_overlap_before": None,
            "keyword_overlap_after": None,
            "tech_score_before": None,
            "tech_score_after": None,
            "action_score_before": None,
            "action_score_after": None,
            "section_score_before": None,
            "section_score_after": None,
            "latency_seconds": result.get("latency_seconds"),
            "error": result.get("error", ""),
            "notes": result.get("notes", "")
        }

    score_comparison = result["score_comparison"]

    return {
        "case_id": result["case_id"],
        "model_name": result["model_name"],
        "task_name": result["task_name"],
        "success": True,
        "before_score": safe_get_score(score_comparison, "before", "overall_score"),
        "after_score": safe_get_score(score_comparison, "after", "overall_score"),
        "score_delta": float(score_comparison.get("score_delta", 0.0)),
        "keyword_overlap_before": safe_get_score(score_comparison, "before", "keyword_overlap_score"),
        "keyword_overlap_after": safe_get_score(score_comparison, "after", "keyword_overlap_score"),
        "tech_score_before": safe_get_score(score_comparison, "before", "tech_keyword_score"),
        "tech_score_after": safe_get_score(score_comparison, "after", "tech_keyword_score"),
        "action_score_before": safe_get_score(score_comparison, "before", "action_verb_score"),
        "action_score_after": safe_get_score(score_comparison, "after", "action_verb_score"),
        "section_score_before": safe_get_score(score_comparison, "before", "section_coverage_score"),
        "section_score_after": safe_get_score(score_comparison, "after", "section_coverage_score"),
        "latency_seconds": result.get("latency_seconds"),
        "error": "",
        "notes": result.get("notes", "")
    }


def run_batch_evaluation(
    cases: List[Dict[str, Any]],
    model_names: List[str]
) -> pd.DataFrame:
    """
    Run a batch evaluation over a list of cases and one or more models.

    Returns a DataFrame with one row per (case, model) result.
    """
    rows = []

    if not cases:
        print("No cases provided for batch evaluation.")
        return pd.DataFrame()

    for case in cases:
        case_id = case.get("case_id", "unknown_case")
        task_name = case.get("task_name", DEFAULT_TASK)

        print(f"\nRunning case: {case_id} | task: {task_name}")

        for model_name in model_names:
            print(f"  -> model: {model_name}")
            result = run_single_case(case, model_name=model_name)
            row = build_result_row(result)
            rows.append(row)

    return pd.DataFrame(rows)


def save_results_dataframe(df: pd.DataFrame, file_path: Path) -> None:
    """
    Save evaluation results to CSV.
    """
    if df.empty:
        print("No results to save.")
        return

    file_path.parent.mkdir(exist_ok=True)
    df.to_csv(file_path, index=False)
    print(f"Saved results to: {file_path}")


def summarize_results(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create a compact summary table grouped by model.
    """
    if df.empty:
        return pd.DataFrame()

    successful_df = df[df["success"] == True].copy()

    if successful_df.empty:
        return pd.DataFrame()

    summary = successful_df.groupby("model_name").agg(
        runs=("case_id", "count"),
        avg_before_score=("before_score", "mean"),
        avg_after_score=("after_score", "mean"),
        avg_score_delta=("score_delta", "mean"),
        avg_latency_seconds=("latency_seconds", "mean")
    ).reset_index()

    summary = summary.round(2)
    return summary

# 11. Save and Review Results

## Save and Review Results

This section saves the evaluation results and creates simple review tables to help compare model performance.

The goal is to make the experimental outputs easier to inspect, summarize, and later include in the final report. In particular, this section helps answer questions such as:

- Which model improved ATS-alignment score more often?
- Which model had the higher average score improvement?
- Were there any failed runs or parsing issues?
- How often did a model make no improvement or perform worse?
- Which cases should be reviewed manually for trustworthiness concerns?

These review tables are useful for both debugging and report writing.

In [14]:
# Save and review evaluation results for the PolishCV project

def add_review_flags(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add simple review flags to help identify cases that need inspection.
    """
    if df.empty:
        return df.copy()

    reviewed_df = df.copy()

    reviewed_df["needs_manual_review"] = (
        (reviewed_df["success"] == False) |
        (reviewed_df["score_delta"].fillna(0) <= 0)
    )

    reviewed_df["score_band_before"] = reviewed_df["before_score"].apply(
        lambda x: score_band(x) if pd.notna(x) else "N/A"
    )
    reviewed_df["score_band_after"] = reviewed_df["after_score"].apply(
        lambda x: score_band(x) if pd.notna(x) else "N/A"
    )

    return reviewed_df


def review_summary_by_model(df: pd.DataFrame) -> pd.DataFrame:
    """
    Summarize key review statistics by model.
    """
    if df.empty:
        return pd.DataFrame()

    summary_rows = []

    for model_name, group in df.groupby("model_name"):
        total_runs = len(group)
        successful_runs = int(group["success"].sum()) if "success" in group.columns else 0
        failed_runs = total_runs - successful_runs

        improved_runs = int((group["score_delta"].fillna(0) > 0).sum())
        no_improvement_runs = int((group["score_delta"].fillna(0) <= 0).sum())
        manual_review_runs = int(group["needs_manual_review"].sum()) if "needs_manual_review" in group.columns else 0

        summary_rows.append({
            "model_name": model_name,
            "total_runs": total_runs,
            "successful_runs": successful_runs,
            "failed_runs": failed_runs,
            "improved_runs": improved_runs,
            "no_or_negative_improvement_runs": no_improvement_runs,
            "needs_manual_review_runs": manual_review_runs,
            "avg_before_score": round(group["before_score"].mean(), 2) if "before_score" in group.columns else None,
            "avg_after_score": round(group["after_score"].mean(), 2) if "after_score" in group.columns else None,
            "avg_score_delta": round(group["score_delta"].mean(), 2) if "score_delta" in group.columns else None,
            "avg_latency_seconds": round(group["latency_seconds"].mean(), 2) if "latency_seconds" in group.columns else None
        })

    return pd.DataFrame(summary_rows)


def get_cases_for_manual_review(df: pd.DataFrame) -> pd.DataFrame:
    """
    Filter rows that should be manually reviewed.
    """
    if df.empty or "needs_manual_review" not in df.columns:
        return pd.DataFrame()

    columns_to_show = [
        "case_id",
        "model_name",
        "task_name",
        "success",
        "before_score",
        "after_score",
        "score_delta",
        "latency_seconds",
        "error",
        "notes"
    ]

    existing_columns = [col for col in columns_to_show if col in df.columns]
    return df[df["needs_manual_review"] == True][existing_columns].copy()


def compare_models_side_by_side(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create a side-by-side comparison of OpenAi vs Llama by case and task.
    """
    if df.empty:
        return pd.DataFrame()

    successful_df = df[df["success"] == True].copy()
    if successful_df.empty:
        return pd.DataFrame()

    pivot_df = successful_df.pivot_table(
        index=["case_id", "task_name"],
        columns="model_name",
        values="score_delta",
        aggfunc="first"
    ).reset_index()

    pivot_df.columns.name = None
    return pivot_df


# Add review flags
reviewed_results_df = add_review_flags(batch_results_df)

# Save reviewed results
save_results_dataframe(reviewed_results_df, EVAL_RESULTS_PATH)

# Build review tables
model_review_summary_df = review_summary_by_model(reviewed_results_df)
manual_review_df = get_cases_for_manual_review(reviewed_results_df)
side_by_side_df = compare_models_side_by_side(reviewed_results_df)

print("Model Review Summary")
display(model_review_summary_df)

print("Cases Needing Manual Review")
display(manual_review_df)

print("Side-by-Side Score Delta Comparison")
display(side_by_side_df)

NameError: name 'batch_results_df' is not defined